In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Task 1: Write your code here:
food_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(food_path)

In [ ]:
# Task 2: Write your code here:
print(f"Dataset shape: {df_food.shape}")
df_food.head()

In [ ]:
# Task 3: Write your code here:
df_food.info()

In [ ]:
# Task 4: Write your code here:
df_food.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Deivery Time Distribution')
plt.xlabel('Delivery Time (Minutes)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_food.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:
print("Missing values:")
print(df_food.isnull().sum())

missing_percentage = (df_food.isnull().sum() / len(df_food)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
#Task 2: Continued

#Create new clean dataframe
cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs', 'Delivery_Time']
df_clean = df_food[cols].copy()

In [ ]:
#Handling missing values

df_clean['Weather'] = df_clean['Weather'].fillna(df_clean['Weather'].mode()[0])
df_clean['Time_of_Day'] = df_clean['Time_of_Day'].fillna(df_clean['Time_of_Day'].mode()[0])


df_clean = df_clean.dropna(subset=['Delivery_Time', 'Courier_Experience_yrs', 'Traffic_Level'])

print("Missing values remaining:", df_clean.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
categorical_cols = [ 'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))


df_clean.head()

In [ ]:
# Task 5: Write your code here:
# Scale features - fit on train, transform both

features = df_clean.columns.drop('Delivery_Time')
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(features)


In [ ]:
# Task 6: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df_clean, "Delivery_Time")

#The target is almost normally distributed, not heavily skewed.

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df_clean['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% for test, remaining 80% for train
    random_state=42,      # reproducible output
    shuffle=True,         # representative splits
    stratify=y            # preserve class distribution
)

model = {"Random Forest Regressor": RandomForestRegressor(n_estimators=100)}

n_splits=5
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]


# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Calculate metrics
mae = sklearn_mae(y_test, y_pred)


In [ ]:
# Task 1: Write your code here:
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
# Plot for Linear Regression Predictions vs. Ground Truth
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], 'r--', linewidth=2)
plt.xlabel("Actual y_test (Ground Truth)")
plt.ylabel("Predicted y_pred (Linear Regression)")
plt.title("Linear Regression: Predictions vs. Ground Truth")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: